In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!pip install -e .

/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 235 (delta 87), reused 205 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (235/235), 154.35 KiB | 7.35 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/RiskAware-Complaints-Engine
Obtaining file:///content/RiskAware-Complaints-Engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for risk-aware-complaints-engine (pyproject.toml) ... done
  Created wheel for risk-aware-complaints-engine: filename=risk_aware_complaints_engine-0.1.0-0.editable-py3-none-any.whl size=2362 sha256=a35558fc121c7f84ff0d865514424d7e26c875f26532a7f95a6b0dff8dcf22ef
  Stored in directory: /tmp/pip-ephem-wheel-

In [10]:
import json, joblib, pandas as pd,os, shutil, zipfile
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive, files

In [3]:
drive.mount('/content/drive')

src = "/content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv"
dst = "/content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv"
os.makedirs("/content/RiskAware-Complaints-Engine/data/raw", exist_ok=True)
shutil.copy(src, dst)
print(os.path.exists(dst), dst)

Mounted at /content/drive
True /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv


In [4]:
!PYTHONPATH=src python src/risk_aware/data/prepare.py
!PYTHONPATH=src python scripts/train_category.py
!PYTHONPATH=src python scripts/evaluate_category.py

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
Category training completed.
Macro-F1: 0.2997
Accuracy: 0.5400
Artifacts saved to: artifacts/category
Validation metrics saved to: reports/metrics/category_metrics_val.json
category_macro_f1: 0.280042
Saved metrics to: reports/metrics/category_metrics_test.json


In [5]:
test = pd.read_csv("data/processed/test.csv")
model = joblib.load("artifacts/category/model.joblib")

x = test["consumer_complaint_narrative"].fillna("").tolist()
y = test["category"].astype(str)
pred = model.predict(x)

print("test_accuracy =", round(accuracy_score(y, pred), 6))
print("test_macro_f1 =", round(f1_score(y, pred, average="macro"), 6))
print("test_weighted_f1 =", round(f1_score(y, pred, average="weighted"), 6))
print("test_classes =", y.nunique())

test_accuracy = 0.527293
test_macro_f1 = 0.280042
test_weighted_f1 = 0.527807
test_classes = 87


In [8]:
bundle = Path("tfidf_baseline_v_current_bundle.zip")
paths = [
    Path("artifacts/category"),
    Path("data/processed"),
    Path("reports/metrics/category_metrics_val.json"),
    Path("reports/metrics/category_metrics_test.json"),
    Path("configs/base.yaml"),
    Path("configs/category.yaml"),
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.is_file():
            z.write(p, arcname=str(p))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f))

print("Created:", bundle.resolve())

Created: /content/RiskAware-Complaints-Engine/tfidf_baseline_v_current_bundle.zip


In [11]:
files.download("/content/RiskAware-Complaints-Engine/tfidf_baseline_v_current_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>